# Fine-Tune Evidence-Aware RAG Synthesizer (v2)

This notebook uses QLoRA (4-bit quantization) to fine-tune `Qwen1.5-0.5B-Chat` as a grounded RAG Synthesizer.
**Improvements over v1:** Full-sentence grounded outputs, multi-epoch training, ROUGE-based auto-evaluation loop.

In [ ]:
%pip install datasets trl peft bitsandbytes rouge-score

In [ ]:
import os
import torch
import json
from torch.utils.data import DataLoader
from tqdm import tqdm
from datasets import load_dataset
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig, get_linear_schedule_with_warmup
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training, PeftModel
from rouge_score import rouge_scorer

os.environ['CUDA_MODULE_LOADING'] = 'EAGER'
if torch.cuda.is_available():
    print(f'CUDA: {torch.cuda.get_device_name(0)} | Memory: {torch.cuda.get_device_properties(0).total_mem / 1e9:.2f} GB')
else:
    print('WARNING: No GPU detected, training will be very slow!')


### Load Grounded Training Data

In [ ]:
# Load the improved grounded-answer dataset
dataset = load_dataset('json', data_files={
    'train': 'D:/RES_cove/adaptive-evidence-rag/data/finetune/train_grounded.jsonl',
    'val': 'D:/RES_cove/adaptive-evidence-rag/data/finetune/val_grounded.jsonl'
})
print(dataset)
print(f'\nSample output: {dataset["train"][0]["output"][:100]}')


### Load Model & Quantization

In [ ]:
model_name = "Qwen/Qwen1.5-0.5B-Chat"
quantization_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True
)
print("Loading Tokenizer...")
tokenizer = AutoTokenizer.from_pretrained(model_name, trust_remote_code=True)
tokenizer.pad_token = tokenizer.eos_token
print("Loading 4-bit Model...")
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    quantization_config=quantization_config,
    device_map="auto",
    trust_remote_code=True,
    dtype=torch.float16,
)
print(f"Model dtype: {model.dtype}")
model = prepare_model_for_kbit_training(model)


### LoRA Configuration (r=16 for better capacity)

In [ ]:
peft_config = LoraConfig(
    r=16,  # Increased from 8 for better learning capacity
    lora_alpha=32,
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"]
)

model = get_peft_model(model, peft_config)
model.print_trainable_parameters()


### Multi-Epoch Training with ROUGE Auto-Stop

Trains for up to 5 epochs. After each epoch, evaluates ROUGE-L on test questions.
Stops early if ROUGE-L >= 0.30 (good for a 0.5B model).

In [ ]:
# === Tokenization ===
def tokenize_function(examples):
    texts = [f"{inst}{out}" for inst, out in zip(examples['instruction'], examples['output'])]
    return tokenizer(texts, truncation=True, max_length=512, padding='max_length')

print("Tokenizing datasets...")
tokenized_train = dataset['train'].map(tokenize_function, batched=True, remove_columns=dataset['train'].column_names)
tokenized_val = dataset['val'].map(tokenize_function, batched=True, remove_columns=dataset['val'].column_names)
tokenized_train.set_format(type='torch', columns=['input_ids', 'attention_mask'])
tokenized_val.set_format(type='torch', columns=['input_ids', 'attention_mask'])

train_loader = DataLoader(tokenized_train, batch_size=1, shuffle=True)
val_loader = DataLoader(tokenized_val, batch_size=1, shuffle=False)

# === Test Questions for ROUGE Evaluation ===
TEST_QA = [
    {
        "prompt": 'You are an advanced Evidence-Aware AI Assistant.\nAnswer the following user question using ONLY the provided verified evidence.\nIf the evidence does not contain the answer, say "I don\'t have enough verified evidence to answer this."\n\nEvidence:\n[Doc 1] The transformer architecture was introduced in the paper "Attention Is All You Need" by Ashish Vaswani et al. at Google in 2017.\n\nQuestion:\nWho invented the transformer architecture?\n\nAnswer:',
        "reference": "Based on the provided evidence, the transformer architecture was invented by Ashish Vaswani et al. at Google in 2017."
    },
    {
        "prompt": 'You are an advanced Evidence-Aware AI Assistant.\nAnswer the following user question using ONLY the provided verified evidence.\nIf the evidence does not contain the answer, say "I don\'t have enough verified evidence to answer this."\n\nEvidence:\n[Doc 1] Python is a high-level programming language created by Guido van Rossum and first released in 1991.\n\nQuestion:\nWho created Python?\n\nAnswer:',
        "reference": "Based on the provided evidence, Python was created by Guido van Rossum."
    },
    {
        "prompt": 'You are an advanced Evidence-Aware AI Assistant.\nAnswer the following user question using ONLY the provided verified evidence.\nIf the evidence does not contain the answer, say "I don\'t have enough verified evidence to answer this."\n\nEvidence:\n[Doc 1] The Great Wall of China is a series of fortifications built along the northern borders of China to protect against various nomadic groups.\n\nQuestion:\nWhat is the purpose of the Great Wall of China?\n\nAnswer:',
        "reference": "According to the verified evidence, the Great Wall of China was built to protect against various nomadic groups."
    },
]

def evaluate_rouge(model, tokenizer, test_qa):
    """Run inference on test questions and compute average ROUGE-L."""
    scorer = rouge_scorer.RougeScorer(['rouge1', 'rougeL'], use_stemmer=True)
    model.eval()
    scores = []
    for qa in test_qa:
        inputs = tokenizer(qa['prompt'], return_tensors='pt').to(model.device)
        with torch.no_grad():
            outputs = model.generate(**inputs, max_new_tokens=80, temperature=0.1, do_sample=True)
        generated = tokenizer.decode(outputs[0][inputs['input_ids'].shape[1]:], skip_special_tokens=True)
        result = scorer.score(qa['reference'], generated)
        scores.append(result['rougeL'].fmeasure)
    model.train()
    return sum(scores) / len(scores)

# === Training Config ===
MAX_EPOCHS = 5
TARGET_ROUGE = 0.30
ACCUMULATION_STEPS = 4
LR = 2e-4

optimizer = torch.optim.AdamW(model.parameters(), lr=LR)
steps_per_epoch = len(train_loader)
total_steps = steps_per_epoch * MAX_EPOCHS
scheduler = get_linear_schedule_with_warmup(optimizer, num_warmup_steps=50, num_training_steps=total_steps)

# === Training Loop ===
print(f"Starting training: up to {MAX_EPOCHS} epochs, {steps_per_epoch} steps/epoch")
print(f"Target ROUGE-L: {TARGET_ROUGE}")

epoch_history = []
global_step = 0

for epoch in range(1, MAX_EPOCHS + 1):
    model.train()
    epoch_loss = 0.0
    pbar = tqdm(train_loader, desc=f"Epoch {epoch}/{MAX_EPOCHS}")
    
    for batch in pbar:
        input_ids = batch['input_ids'].to(model.device)
        attention_mask = batch['attention_mask'].to(model.device)
        labels = input_ids.clone()
        labels[attention_mask == 0] = -100
        
        with torch.amp.autocast('cuda', dtype=torch.float16):
            outputs = model(input_ids=input_ids, attention_mask=attention_mask, labels=labels)
            loss = outputs.loss / ACCUMULATION_STEPS
        
        loss.backward()
        epoch_loss += loss.item() * ACCUMULATION_STEPS
        
        if (global_step + 1) % ACCUMULATION_STEPS == 0:
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=0.3)
            optimizer.step()
            scheduler.step()
            optimizer.zero_grad()
        
        global_step += 1
        pbar.set_postfix({'loss': f'{loss.item() * ACCUMULATION_STEPS:.4f}'})
        
        if global_step % 100 == 0:
            print(f"Step {global_step} | Loss: {loss.item() * ACCUMULATION_STEPS:.4f} | LR: {scheduler.get_last_lr()[0]:.2e}")
    
    avg_loss = epoch_loss / len(train_loader)
    
    # Save checkpoint
    ckpt_path = f"../models/qwen-evidence-rag-lora-v2/epoch-{epoch}"
    os.makedirs(ckpt_path, exist_ok=True)
    model.save_pretrained(ckpt_path)
    
    # Evaluate ROUGE
    rouge_l = evaluate_rouge(model, tokenizer, TEST_QA)
    epoch_history.append({'epoch': epoch, 'loss': avg_loss, 'rouge_l': rouge_l})
    
    print(f"\n{'='*50}")
    print(f"Epoch {epoch} Complete | Avg Loss: {avg_loss:.4f} | ROUGE-L: {rouge_l:.4f}")
    print(f"{'='*50}")
    
    if rouge_l >= TARGET_ROUGE:
        print(f"TARGET REACHED! ROUGE-L {rouge_l:.4f} >= {TARGET_ROUGE}")
        break
    else:
        print(f"ROUGE-L {rouge_l:.4f} < {TARGET_ROUGE}, continuing training...")

# Save final model
final_path = "../models/qwen-evidence-rag-lora-final"
os.makedirs(final_path, exist_ok=True)
model.save_pretrained(final_path)
print(f"\nFinal model saved to {final_path}")
print(f"\nTraining History:")
for h in epoch_history:
    print(f"  Epoch {h['epoch']}: Loss={h['loss']:.4f}, ROUGE-L={h['rouge_l']:.4f}")


### Training Progress & Evaluation Graphs

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

epochs = [h['epoch'] for h in epoch_history]
losses = [h['loss'] for h in epoch_history]
rouges = [h['rouge_l'] for h in epoch_history]

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# Loss curve
ax1.plot(epochs, losses, 'o-', color='#ff6b6b', linewidth=2, markersize=8)
ax1.set_title('Training Loss per Epoch', fontsize=14, fontweight='bold')
ax1.set_xlabel('Epoch')
ax1.set_ylabel('Average Loss')
ax1.grid(True, alpha=0.3)

# ROUGE-L curve
ax2.plot(epochs, rouges, 'o-', color='#51cf66', linewidth=2, markersize=8)
ax2.axhline(y=TARGET_ROUGE, color='red', linestyle='--', label=f'Target ({TARGET_ROUGE})')
ax2.set_title('ROUGE-L Score per Epoch', fontsize=14, fontweight='bold')
ax2.set_xlabel('Epoch')
ax2.set_ylabel('ROUGE-L F1')
ax2.legend()
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('../models/training_progress.png', dpi=150)
plt.show()
print('Training progress saved to models/training_progress.png')


### Final Inference Test

In [ ]:
# Test the final model
model.eval()
for qa in TEST_QA:
    inputs = tokenizer(qa['prompt'], return_tensors='pt').to(model.device)
    with torch.no_grad():
        outputs = model.generate(**inputs, max_new_tokens=80, temperature=0.1, do_sample=True)
    generated = tokenizer.decode(outputs[0][inputs['input_ids'].shape[1]:], skip_special_tokens=True)
    q = qa['prompt'].split('Question:')[-1].split('Answer:')[0].strip()
    print(f'Q: {q}')
    print(f'Expected: {qa["reference"]}')
    print(f'Generated: {generated}')
    print()
